# 05 - RQ4: Is TabNet's Attention Actually Trustworthy?

**Notebook version:** v33 -- 2026-08-30

- Train TabNet and a control MLP (same depth/width, no attention) on the reduced feature set
- Extract TabNet attention weights; use Integrated Gradients (Sundararajan et al., 2017) for the MLP control
- Compute Spearman correlation of each model's ranking against the SHAP ranking
- Compare TabNet-vs-SHAP agreement to MLP-vs-SHAP agreement to isolate attention's contribution


In [ ]:
# --- Colab setup: run this cell first if you opened this notebook from GitHub in Colab ---
# If you're running locally in Jupyter from the notebooks/ folder, this cell does nothing.
# Safe to re-run: always anchors to /content so repeated runs never create nested clones.
# git pull always runs (cheap, ~seconds) so code is never stale even if Colab's
# 'Restart session' left /content on disk from an earlier session -- only
# pip install (the actual slow part) is skipped via the session marker.
import os
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/WJPsystems/secom-explainable-vm.git"
REPO_NAME = "secom-explainable-vm"
SETUP_MARKER = Path("/content/.secom_setup_done")

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        !git clone "{REPO_URL}"
        SETUP_MARKER.unlink(missing_ok=True)  # fresh clone -- force full setup below

    # Always pull -- cheap, and guarantees code is current even if /content
    # persisted on disk from an earlier session (e.g. Colab 'Restart session'
    # rather than a full 'Disconnect and delete runtime').
    os.chdir(f"/content/{REPO_NAME}")
    !git pull
    os.chdir("/content")

    os.chdir(f"/content/{REPO_NAME}/notebooks")

    already_setup = SETUP_MARKER.exists()
    if not already_setup:
        !pip install -q -r ../requirements.txt
        SETUP_MARKER.touch()
        setup_note = "Ran pip install (git pull always runs regardless)."
    else:
        setup_note = "Skipped pip install -- already done earlier this session. git pull always ran above."

    commit_info = subprocess.run(
        ["git", "log", "-1", "--format=%h %ci"], capture_output=True, text=True
    ).stdout.strip()
    print(f"Colab setup complete. Working directory: {os.getcwd()}")
    print(setup_note)
    print(f"Repo commit: {commit_info}")
    print("Compare this commit hash against GitHub's latest commit to confirm you're current.")
else:
    print("Not running in Colab -- assuming local Jupyter launched from the notebooks/ folder.")

In [ ]:
import sys

try:
    import google.colab
    _SRC_PATH = "/content/secom-explainable-vm/src"
except ImportError:
    _SRC_PATH = "../src"
if _SRC_PATH not in sys.path:
    sys.path.append(_SRC_PATH)

import numpy as np
import pandas as pd

from preprocessing import load_raw, screen_missingness, screen_variance, impute_median
from artifacts import load_json
from explainability import per_instance_agreement

X, y = load_raw()
X = impute_median(screen_variance(screen_missingness(X)))

rq3_summary = load_json("rq3_summary")
final_features = rq3_summary["final_features"]
print(f"Using RQ3's reduced feature set ({len(final_features)}): {final_features}")

holdout_split = load_json("rq1_holdout_split")
train_idx, test_idx = holdout_split["train_idx"], holdout_split["test_idx"]
X_train_ho = X.iloc[train_idx][final_features]
X_test_ho = X.iloc[test_idx][final_features]
y_train_ho, y_test_ho = y.iloc[train_idx], y.iloc[test_idx]
print(f"Train: {len(X_train_ho)}, Test (held out): {len(X_test_ho)}")


## Step 1: Train TabNet and a plain MLP control on the same reduced feature set

Both trained on `final_features` only (RQ3's reduced set) so the RQ4
comparison is scoped consistently with the rest of the pipeline. The MLP
control is identical in spirit to TabNet (a feedforward neural net) but
has no attention mechanism -- its role is to isolate how much of any
SHAP-agreement is attributable to attention specifically, vs. just being
a neural network in general.

In [ ]:
import torch
import torch.nn as nn
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import StandardScaler

# Feature scaling is not optional here -- SECOM sensors span wildly
# different numeric ranges (the same issue that broke LASSO earlier in this
# pipeline). Feeding unscaled features into the MLP caused a real, confirmed
# failure: training loss exploded to ~174 and flatlined for 100+ epochs (a
# saturated/dead network), producing garbage SHAP/IG attributions and an
# artifactual 0.0000 agreement score. TabNet handles this internally; a raw
# PyTorch MLP does not.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_ho).astype("float32")
X_test_scaled = scaler.transform(X_test_ho).astype("float32")

X_train_arr = X_train_ho.values.astype("float32")  # TabNet: unscaled, handles it internally
y_train_arr = y_train_ho.values
X_test_arr = X_test_ho.values.astype("float32")

torch.manual_seed(42)

tabnet = TabNetClassifier(seed=42, verbose=0)
tabnet.fit(X_train_arr, y_train_arr, weights=1, max_epochs=60, patience=10, batch_size=128)
print("TabNet trained.")

class MLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )  # raw logits -- no Sigmoid here; BCEWithLogitsLoss is more numerically
           # stable than Sigmoid+BCELoss, and was part of the actual fix, not just
           # the scaling. Sigmoid is applied explicitly wherever a probability is
           # actually needed (below, and in Steps 2-3).
    def forward(self, x):
        return self.net(x)

mlp = MLP(len(final_features))
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001)  # lowered from 0.01 -- part of the fix
loss_fn = nn.BCEWithLogitsLoss(reduction="none")

pos_weight_ratio = (y_train_arr == 0).sum() / max((y_train_arr == 1).sum(), 1)
sample_weights = np.where(y_train_arr == 1, pos_weight_ratio, 1.0).astype("float32")

X_train_t = torch.tensor(X_train_scaled)
y_train_t = torch.tensor(y_train_arr.astype("float32")).unsqueeze(1)
weights_t = torch.tensor(sample_weights).unsqueeze(1)

for epoch in range(200):
    optimizer.zero_grad()
    out = mlp(X_train_t)
    loss = (loss_fn(out, y_train_t) * weights_t).mean()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(mlp.parameters(), max_norm=1.0)  # safety net against future divergence
    optimizer.step()
    if epoch % 50 == 0:
        with torch.no_grad():
            train_acc = ((torch.sigmoid(out) >= 0.5).float() == y_train_t).float().mean().item()
        print(f"  MLP epoch {epoch}: loss={loss.item():.4f}, train_acc={train_acc:.4f}")

with torch.no_grad():
    final_train_acc = ((torch.sigmoid(mlp(X_train_t)) >= 0.5).float() == y_train_t).float().mean().item()
print(f"MLP trained. Final train accuracy: {final_train_acc:.4f}")
if final_train_acc < 0.5:
    print(
        "WARNING: MLP train accuracy is at or below chance -- this suggests the "
        "model has NOT trained successfully (same symptom as the divergence bug "
        "this fix addresses). Do not trust downstream SHAP/IG comparisons if you "
        "see this warning."
    )


## Step 2: Extract each model's own explanation

TabNet's `.explain()` is its native attention-derived per-instance feature
importance. Integrated Gradients (Sundararajan, Taly, & Yan, 2017) plays
the same role for the MLP control, since it has no attention layer to
extract -- see docs/synopsis.docx, "Solution to RQ4" for why IG rather
than raw saliency.

In [ ]:
from captum.attr import IntegratedGradients

tabnet_explain, _ = tabnet.explain(X_test_arr)
print(f"TabNet attention explanations: {tabnet_explain.shape}")

# Diagnostic: TabNet's attention output is a sparsemax-derived mask, which is
# typically NON-NEGATIVE (magnitude of feature use, not direction of
# contribution) -- unlike SHAP and Integrated Gradients, which are SIGNED
# (direction and magnitude of contribution). If confirmed non-negative here,
# correlating it against signed SHAP values in Step 4 partly tests a
# representational mismatch, not just "does the ranking agree" -- checked
# explicitly rather than assumed, since it directly affects how to interpret
# the significance test result.
print(f"TabNet attention value range: min={tabnet_explain.min():.6f}, max={tabnet_explain.max():.6f}")
if tabnet_explain.min() >= 0:
    print("Confirmed: TabNet's attention output is non-negative (magnitude-only, no direction).")

# Attribute on the SAME scaled inputs the MLP was trained on, and wrap with
# sigmoid so IG explains the probability output (matching SHAP's scale below),
# not the raw logit.
mlp_prob = lambda x: torch.sigmoid(mlp(x))
ig = IntegratedGradients(mlp_prob)
X_test_scaled_t = torch.tensor(X_test_scaled, requires_grad=True)
baseline = torch.zeros_like(X_test_scaled_t)
ig_attributions = ig.attribute(X_test_scaled_t, baseline, target=0).detach().numpy()
print(f"MLP Integrated Gradients: {ig_attributions.shape}")
print(f"MLP Integrated Gradients value range: min={ig_attributions.min():.6f}, max={ig_attributions.max():.6f}")


## Step 3: SHAP explanations for both models

Neither TabNet nor the MLP is a tree model, so `TreeExplainer` doesn't
apply here -- `KernelExplainer` is used instead (model-agnostic, works on
any `predict_proba`-like function), per the synopsis's discussion of
SHAP's approximate mode for non-tree models. `KernelExplainer` is
computationally expensive relative to `TreeExplainer` (no exact/fast path
for neural nets), so this is deliberately run on a subsample of the
held-out set (`N_EXPLAIN`), not all 314 wafers -- increase it if runtime
allows, but expect this cell to be the slowest one in the notebook.

In [ ]:
import shap

N_BACKGROUND = 50
N_EXPLAIN = 40

# TabNet: unscaled background/explain set (trained on unscaled data)
background_tabnet = shap.sample(X_train_ho, N_BACKGROUND, random_state=42)
X_explain_tabnet = X_test_ho.iloc[:N_EXPLAIN]

tabnet_predict_fn = lambda x: tabnet.predict_proba(x.astype("float32"))[:, 1]
explainer_tabnet = shap.KernelExplainer(tabnet_predict_fn, background_tabnet)

# shap.sample()'s background selection is seeded (random_state=42 above),
# but KernelExplainer.shap_values()'s own Monte Carlo coalition sampling is
# NOT seeded internally -- it draws from numpy's global random state. This
# caused real, confirmed run-to-run drift in RQ4's reported numbers across
# four separate real runs (qualitative conclusion stable each time, exact
# figures were not). Resetting the global seed immediately before each
# shap_values() call, verified on synthetic data to produce bit-identical
# results across repeated calls, closes this reproducibility gap.
np.random.seed(42)
shap_tabnet = np.array(explainer_tabnet.shap_values(X_explain_tabnet, nsamples=100))
print(f"SHAP (TabNet) computed: {shap_tabnet.shape}")

# MLP: SCALED background/explain set -- must match what the MLP was trained
# on, using the SAME fitted scaler (not a freshly-fit one, which would leak
# test-set statistics into the transform).
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=final_features, index=X_train_ho.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=final_features, index=X_test_ho.index)
background_mlp = shap.sample(X_train_scaled_df, N_BACKGROUND, random_state=42)
X_explain_mlp = X_test_scaled_df.iloc[:N_EXPLAIN]

def mlp_predict_fn(x):
    with torch.no_grad():
        return torch.sigmoid(mlp(torch.tensor(x.astype("float32")))).numpy().flatten()

explainer_mlp = shap.KernelExplainer(mlp_predict_fn, background_mlp)

# Re-seed here too -- a single seed() call before the TabNet computation
# above would be consumed by that call's own random draws, leaving this
# call's sampling unseeded again if not reset explicitly.
np.random.seed(42)
shap_mlp = np.array(explainer_mlp.shap_values(X_explain_mlp, nsamples=100))
print(f"SHAP (MLP) computed: {shap_mlp.shape}")


## Step 4: RQ4's actual answer -- does attention agree with SHAP better than IG does?

Per-instance Spearman correlation and top-3 feature-set agreement between
each model's own explanation and its own SHAP values, restricted to the
same `N_EXPLAIN` instances both were computed on.

In [ ]:
from explainability import paired_agreement_significance_test

tabnet_agreement = per_instance_agreement(shap_tabnet, tabnet_explain[:N_EXPLAIN], top_k=3)
mlp_agreement = per_instance_agreement(shap_mlp, ig_attributions[:N_EXPLAIN], top_k=3)

print("=== TabNet: SHAP vs. attention agreement ===")
print(f"  Mean Spearman correlation: {tabnet_agreement['mean_correlation']:.4f}")
print(f"  Median Spearman correlation: {tabnet_agreement['median_correlation']:.4f}")
print(f"  Top-3 feature-set agreement rate: {tabnet_agreement['top_k_agreement_rate']:.2%}")

print("\n=== MLP control: SHAP vs. Integrated Gradients agreement ===")
print(f"  Mean Spearman correlation: {mlp_agreement['mean_correlation']:.4f}")
print(f"  Median Spearman correlation: {mlp_agreement['median_correlation']:.4f}")
print(f"  Top-3 feature-set agreement rate: {mlp_agreement['top_k_agreement_rate']:.2%}")

# Formal significance test -- RQ4's own H0/Ha require testing whether
# TabNet's and the MLP's per-instance SHAP-agreement correlations genuinely
# differ, not just comparing two means and treating a visible gap as
# self-evidently significant. Paired (not independent-samples), since both
# arrays are computed on the SAME N_EXPLAIN wafers.
sig_test = paired_agreement_significance_test(
    tabnet_agreement["per_instance_correlation"],
    mlp_agreement["per_instance_correlation"],
)
print("\n=== Formal significance test: TabNet vs. MLP SHAP-agreement (paired, N=%d) ===" % sig_test["n_pairs"])
print(f"  Mean difference (TabNet - MLP): {sig_test['mean_diff']:.4f}")
print(f"  Paired t-test: t={sig_test['paired_ttest_statistic']:.4f}, p={sig_test['paired_ttest_pvalue']:.6f}")
print(f"  Wilcoxon signed-rank (robustness check): W={sig_test['wilcoxon_statistic']:.4f}, p={sig_test['wilcoxon_pvalue']:.6f}")

alpha = 0.05
is_significant = sig_test["paired_ttest_pvalue"] < alpha

print("\n=== RQ4 answer ===")
print(f"At alpha = {alpha}: {'REJECT' if is_significant else 'FAIL TO REJECT'} H0(4).")
if is_significant and tabnet_agreement["mean_correlation"] < mlp_agreement["mean_correlation"]:
    print(
        "TabNet's attention agrees with SHAP significantly LESS than the MLP control's "
        "Integrated Gradients does -- this supports Ha(4) and suggests TabNet's built-in "
        "'interpretability' claim does not hold up better than a generic post-hoc "
        "explanation applied to any neural net, which is itself a reportable finding."
    )
elif is_significant:
    print(
        "TabNet's attention agrees with SHAP significantly MORE than the MLP control's "
        "Integrated Gradients does -- this supports Ha(4) in TabNet's favor, some evidence "
        "for its built-in interpretability claim."
    )
else:
    print(
        "The difference between TabNet's and the MLP's SHAP-agreement is not statistically "
        "significant at alpha=0.05 -- fail to reject H0(4). Despite the visible gap in mean "
        "correlation, this sample (N=%d wafers) does not provide sufficient evidence to "
        "conclude the two differ." % sig_test["n_pairs"]
    )


## Step 5: Supplementary check -- is this a fair, like-for-like comparison?

TabNet's `.explain()` output is a sparsemax-derived attention mask, which is
confirmed non-negative in Step 2 above (magnitude of feature use only, no
direction). SHAP values and Integrated Gradients, by contrast, are SIGNED
(direction and magnitude of contribution). Correlating a non-negative array
against a signed array in Step 4 partly tests a representational mismatch,
not purely "does the importance ranking agree" -- confirmed on synthetic
data that this mismatch alone can make a genuinely-agreeing pair of signals
look like they disagree (a raw correlation of -0.06 when the true magnitude-
only agreement was 0.82). This step re-runs the same comparison using
absolute values on BOTH sides, putting TabNet and the MLP control on equal,
sign-agnostic footing, to test whether Step 4's conclusion holds up.

In [ ]:
tabnet_agreement_abs = per_instance_agreement(
    np.abs(shap_tabnet), np.abs(tabnet_explain[:N_EXPLAIN]), top_k=3
)
mlp_agreement_abs = per_instance_agreement(
    np.abs(shap_mlp), np.abs(ig_attributions[:N_EXPLAIN]), top_k=3
)

print("=== Magnitude-only (sign-agnostic) agreement ===")
print(f"TabNet |attention| vs |SHAP|: mean correlation = {tabnet_agreement_abs['mean_correlation']:.4f}")
print(f"MLP |IG| vs |SHAP|:           mean correlation = {mlp_agreement_abs['mean_correlation']:.4f}")

sig_test_abs = paired_agreement_significance_test(
    tabnet_agreement_abs["per_instance_correlation"],
    mlp_agreement_abs["per_instance_correlation"],
)
print(f"\nPaired t-test (magnitude-only): t={sig_test_abs['paired_ttest_statistic']:.4f}, "
      f"p={sig_test_abs['paired_ttest_pvalue']:.6f}")
print(f"Wilcoxon (magnitude-only):      W={sig_test_abs['wilcoxon_statistic']:.4f}, "
      f"p={sig_test_abs['wilcoxon_pvalue']:.6f}")

print("\n=== Interpretation ===")
abs_still_significant = sig_test_abs["paired_ttest_pvalue"] < 0.05
abs_gap_shrunk = abs(tabnet_agreement_abs["mean_correlation"] - mlp_agreement_abs["mean_correlation"]) < \
                 abs(tabnet_agreement["mean_correlation"] - mlp_agreement["mean_correlation"])
if abs_still_significant and not abs_gap_shrunk:
    print(
        "The gap persists on a fair, sign-agnostic basis and remains significant -- "
        "Step 4's conclusion is ROBUST to the signed-vs-unsigned representational "
        "concern, not an artifact of it."
    )
elif abs_still_significant and abs_gap_shrunk:
    print(
        "The gap narrows on a sign-agnostic basis but remains statistically significant -- "
        "some of Step 4's raw effect size was attributable to the representational "
        "mismatch, but a genuine disagreement in magnitude-ranking remains."
    )
else:
    print(
        "The gap is NOT significant on a sign-agnostic basis -- this suggests Step 4's "
        "conclusion was substantially driven by the signed-vs-unsigned representational "
        "mismatch, not a genuine difference in how well each method's magnitude-ranking "
        "agrees with SHAP's. This meaningfully qualifies the RQ4 conclusion and should "
        "be reported alongside it, not omitted."
    )


## Figures: Raw vs. magnitude-only agreement, and per-instance scatter (Visual Evidence for Final Report)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from plotting import style_axes, save_figure, FIGSIZE_STANDARD

# --- Raw vs. magnitude-only agreement, grouped bar chart ---
categories = ["TabNet vs SHAP", "MLP vs SHAP"]
raw_vals = [tabnet_agreement["mean_correlation"], mlp_agreement["mean_correlation"]]
abs_vals = [tabnet_agreement_abs["mean_correlation"], mlp_agreement_abs["mean_correlation"]]
x = np.arange(len(categories))
width = 0.35
fig, ax = plt.subplots(figsize=FIGSIZE_STANDARD)
ax.bar(x - width/2, raw_vals, width, label="Raw (signed)", color="#d62728")
ax.bar(x + width/2, abs_vals, width, label="Magnitude-only", color="#9467bd")
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(categories)
style_axes(ax, title="RQ4: Raw vs. Magnitude-Only SHAP Agreement", ylabel="Mean Spearman Correlation")
ax.legend(fontsize=9)
save_figure(fig, "rq4_raw_vs_magnitude")

# --- Per-instance paired scatter: TabNet vs. MLP agreement with SHAP ---
fig, ax = plt.subplots(figsize=FIGSIZE_STANDARD)
ax.scatter(
    tabnet_agreement["per_instance_correlation"],
    mlp_agreement["per_instance_correlation"],
    color="#1f77b4", alpha=0.7, edgecolor='white', s=50,
)
lims = [-1, 1]
ax.plot(lims, lims, linestyle="--", color="gray", linewidth=1, label="y = x (equal agreement)")
ax.set_xlim(lims); ax.set_ylim(lims)
style_axes(
    ax, title=f"RQ4: Per-Instance SHAP Agreement, TabNet vs. MLP (N={len(tabnet_agreement['per_instance_correlation'])} wafers)",
    xlabel="TabNet correlation with SHAP", ylabel="MLP correlation with SHAP",
)
ax.legend(fontsize=9)
save_figure(fig, "rq4_per_instance_scatter")


## Next steps

- Report both agreement rates and the RQ4 conclusion in the capstone
  write-up
- Note the SHAP sample size limitation (`N_EXPLAIN` wafers, not the full
  314) explicitly if reporting these numbers -- a smaller, faster proxy
  for the full held-out set, not the full set itself

## Export

Run the cell below last to export this notebook to a standalone HTML file.

In [ ]:
# --- Export this notebook to HTML (run this cell last) ---
# Works whether opened live from GitHub in Colab or run locally in Jupyter.
import json
import subprocess

NOTEBOOK_NAME = "05_attention_comparison_rq4"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

export_path = f"{NOTEBOOK_NAME}.ipynb"
live_export_available = False

if IN_COLAB:
    # Colab's own notebook JSON isn't the same file as the clone on disk --
    # this pulls the live, currently-run state (including your outputs)
    # directly from the Colab frontend, so nothing gets missed. This only
    # works when there's an actual live frontend attached (i.e. you're running
    # this cell interactively yourself) -- it returns None instead of raising
    # when run unattended (e.g. via 00_run_all.ipynb's automated execution),
    # so that case is caught explicitly here rather than left to crash with a
    # raw TypeError, which used to make 00_run_all's --allow-errors flag mask
    # *real* failures elsewhere in the notebook, not just this expected one.
    from google.colab import _message
    response = _message.blocking_request('get_ipynb', timeout_sec=30)
    if response is not None:
        ipynb_content = response['ipynb']
        with open(export_path, 'w') as f:
            json.dump(ipynb_content, f)
        live_export_available = True
    else:
        print(
            "No live Colab frontend detected (expected when run via "
            "00_run_all.ipynb) -- skipping the live export. 00_run_all does its "
            "own separate HTML export against the already-executed file instead."
        )

if live_export_available or not IN_COLAB:
    html_output = f"{NOTEBOOK_NAME}.html"
    result = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', export_path, '--output', html_output],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    else:
        print(f"Exported to {html_output}")

    if IN_COLAB and result.returncode == 0:
        from google.colab import files
        files.download(html_output)
